# 02 - Feature engineering
Goal: create new columns the models can use.

## Step 1: Load the clean file

In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/campaigns_clean.csv")
print(df.shape)

(166665, 19)


In [2]:
df_feat = df.copy()

## Step 2: Make a copy for the new features

In [3]:
df_feat["Channel_Used"].value_counts().head(10)

Channel_Used
Email                 8953
Instagram             8858
Facebook              8791
Google                8740
WhatsApp              8705
YouTube               8686
Unknown               8282
YouTube, Instagram    1834
Email, Facebook       1824
WhatsApp, Google      1816
Name: count, dtype: int64

## Step 3: Look at the Channel_Used column
One campaign can use several channels in one cell, like `YouTube, Instagram`.

In [4]:
# 1 if the channel name appears in the text, otherwise 0
df_feat["Channel_Instagram"] = df_feat["Channel_Used"].str.contains("Instagram").astype(int)
df_feat["Channel_Email"] = df_feat["Channel_Used"].str.contains("Email").astype(int)
df_feat["Channel_Google"] = df_feat["Channel_Used"].str.contains("Google").astype(int)
df_feat["Channel_Facebook"] = df_feat["Channel_Used"].str.contains("Facebook").astype(int)
df_feat["Channel_YouTube"] = df_feat["Channel_Used"].str.contains("YouTube").astype(int)
df_feat["Channel_WhatsApp"] = df_feat["Channel_Used"].str.contains("WhatsApp").astype(int)

## Step 4: Multi-label encoding of channels
For each channel we make a 0/1 column. 1 means the campaign used that channel.
One campaign can have several 1s. That is why it is called multi-label.

In [5]:
df_feat["Channel_Count"] = (
    df_feat["Channel_Instagram"]
    + df_feat["Channel_Email"]
    + df_feat["Channel_Google"]
    + df_feat["Channel_Facebook"]
    + df_feat["Channel_YouTube"]
    + df_feat["Channel_WhatsApp"]
)

## Step 5: Count how many channels each campaign used
Add the six 0/1 columns together. `Unknown` gives 0.

In [6]:
df_feat[["Channel_Used", "Channel_Instagram", "Channel_Email", "Channel_Google",
         "Channel_Facebook", "Channel_YouTube", "Channel_WhatsApp", "Channel_Count"]].head(10)

,Channel_Used,Channel_Instagram,Channel_Email,Channel_Google,Channel_Facebook,Channel_YouTube,Channel_WhatsApp,Channel_Count
0,"WhatsApp, YouTube",0,0,0,0,1,1,2
1,YouTube,0,0,0,0,1,0,1
2,"WhatsApp, Google, YouTube",0,0,1,0,1,1,3
3,"YouTube, Facebook, Instagram",1,0,0,1,1,0,3
4,"Facebook, Instagram",1,0,0,1,0,0,2
5,"Email, Instagram",1,1,0,0,0,0,2
6,"YouTube, Google, WhatsApp",0,0,1,0,1,1,3
7,Unknown,0,0,0,0,0,0,0
8,"Facebook, WhatsApp, Email",0,1,0,1,0,1,3
9,Email,0,1,0,0,0,0,1


Check the result on the first 10 rows.

In [7]:
# Start with 0 (Loss) for every row
df_feat["Profit_Flag"] = 0

# ROI above 0 means the campaign earned more than it cost -> Profit (1)
df_feat.loc[df_feat["ROI"] > 0, "Profit_Flag"] = 1

## Step 6: Create the Profit_Flag (our classification target)
ROI above 0 means the campaign earned more than it cost.
- 1 = Profit
- 0 = Loss

In [8]:
# Start with 0 (Loss) for every row
df_feat["Profit_Flag"] = 0

# ROI above 0 means the campaign earned more than it cost -> Profit (1)
df_feat.loc[df_feat["ROI"] > 0, "Profit_Flag"] = 1

Profit_Flag
1    126871
0     39794
Name: count, dtype: int64
Profit_Flag
1    0.761
0    0.239
Name: proportion, dtype: float64


Count how many Profit and Loss campaigns we have.

In [9]:
# Year and Month already hold the date information
df_feat = df_feat.drop(columns=["Date"])

## Step 7: Drop the Date column
Year and Month already hold the date information.

In [10]:
print(df_feat.shape)
print(df_feat.isnull().sum().sum())

(166665, 26)
0


## Step 8: Final check

In [11]:
df_feat.to_csv("../data/processed/campaigns_features.csv", index=False)
print("Saved!")

Saved!
